In [41]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.ar_model import AutoReg
import matplotlib.pyplot as plt
import copy
import json
import os
import random
import time
import zlib
import warnings
from joblib import Parallel, delayed
from pathlib import Path
warnings.filterwarnings('ignore')

# Random seeds
np.random.seed(42)
torch.manual_seed(42)

In [42]:
# LSTM Config
class Config_LSTM:
    use_bic: bool = False
    data_source = Path('..') / 'Data'
    data_files = [
        'AAPL.csv'
    ]
    estimate_type = 'QMLE-Trade'
    value_column = 'Volatility'
    output_root = Path('..') / 'Results'

    lookback_period = 10
    max_bic_lag = 60
    bic_selection_years = 3
    initial_train_years = 5
    validation_years    = 1
    test_months         = 1

    hidden_layers         = [16, 8]
    learning_rate         = 0.001
    batch_size            = 512
    epochs                = 100
    early_stopping_rounds = 10
    n_ensembles           = 5
    random_seed            = 42
    batch_normalization    = False

    # Portable CPU multiprocessing defaults. Override cpu_workers after
    # constructing the config if a particular machine needs a different value.
    cpu_workers             = max(1, min(8, (os.cpu_count() or 2) // 2))
    torch_threads_per_worker = 1
    parallel_verbose         = 0

    lag_candidates = sorted(list(range(1, lookback_period + 1)), reverse=True)
    use_log_transform = True
    target_scaler     = True

    device = torch.device('cpu')
    print(f"Using CPU with {cpu_workers} parallel window workers")

Using CPU with 8 parallel window workers


In [43]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim: int, hidden_layers, dropout: float = 0.0):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_layers = list(hidden_layers)
        self.dropout = float(dropout)

        # Build LSTM stack
        self.lstm_layers = nn.ModuleList()
        self.lstm_layers.append(nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_layers[0],
            num_layers=1,
            batch_first=True
        ))

        for in_size, out_size in zip(self.hidden_layers[:-1], self.hidden_layers[1:]):
            self.lstm_layers.append(nn.LSTM(
                input_size=in_size,
                hidden_size=out_size,
                num_layers=1,
                batch_first=True
            ))

        self.use_dropout = self.dropout > 0.0
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)

        last_hidden = self.hidden_layers[-1]
        self.fc_out = nn.Linear(last_hidden, 1)

        nn.init.kaiming_uniform_(self.fc_out.weight, a=np.sqrt(5))
        if self.fc_out.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.fc_out.weight)
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.fc_out.bias, -bound, bound)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3:
            pass
        else:
            raise ValueError(f"Expected input of shape (N, L) or (N, L, C), got {tuple(x.shape)}")

        for lstm in self.lstm_layers:
            x, _ = lstm(x)

        last = x[:, -1, :]
        if self.use_dropout:
            last = self.dropout_layer(last)

        out = self.fc_out(last)
        return out

In [44]:
# Data loading and panel construction
def load_dacheng_xiu_panel(
    data_directory,
    data_files,
    estimate_type='QMLE-Trade',
    value_column='Volatility'
):
    data_directory = Path(data_directory)
    if not data_directory.is_dir():
        raise FileNotFoundError(f"Data directory does not exist: {data_directory}")
    if not data_files:
        raise ValueError("Add at least one Dacheng Xiu CSV filename to data_files")

    csv_paths = []
    for filename in data_files:
        filename_path = Path(filename)
        if filename_path.suffix == '':
            filename_path = filename_path.with_suffix('.csv')
        csv_path = data_directory / filename_path
        if not csv_path.is_file():
            raise FileNotFoundError(f"Selected CSV does not exist: {csv_path}")
        csv_paths.append(csv_path)

    if len(set(csv_paths)) != len(csv_paths):
        raise ValueError("data_files contains duplicate filenames")

    required_columns = {'Symbol', 'PN', 'Type', 'Date', value_column}
    selected_frames = []

    for csv_path in csv_paths:
        source_data = pd.read_csv(csv_path)
        missing_columns = required_columns.difference(source_data.columns)
        if missing_columns:
            raise ValueError(
                f"{csv_path.name} is missing columns: {sorted(missing_columns)}"
            )

        ticker_data = source_data.loc[
            source_data['Type'].eq(estimate_type),
            ['Symbol', 'Date', value_column]
        ].copy()
        if ticker_data.empty:
            raise ValueError(
                f"{csv_path.name} contains no rows for estimate type '{estimate_type}'"
            )

        ticker_data['Date'] = pd.to_datetime(ticker_data['Date'], errors='raise')
        ticker_data[value_column] = pd.to_numeric(
            ticker_data[value_column], errors='raise'
        )
        selected_frames.append(ticker_data)

    combined_data = pd.concat(selected_frames, ignore_index=True)
    duplicate_mask = combined_data.duplicated(['Date', 'Symbol'], keep=False)
    if duplicate_mask.any():
        duplicate_examples = combined_data.loc[
            duplicate_mask, ['Date', 'Symbol']
        ].drop_duplicates().head(5)
        raise ValueError(
            "Multiple selected observations found for the same symbol/date. "
            f"Examples:\n{duplicate_examples.to_string(index=False)}"
        )

    panel = combined_data.pivot(
        index='Date', columns='Symbol', values=value_column
    ).sort_index()
    panel.index.name = 'date'
    panel.columns.name = None

    print(
        f"Constructed panel from {len(csv_paths)} Dacheng Xiu CSV file(s) "
        f"using {estimate_type}; {value_column} values were not squared"
    )
    print(f"Panel shape: {panel.shape[0]} dates x {panel.shape[1]} symbols")
    print(f"Global date range: {panel.index.min().date()} to {panel.index.max().date()}")
    return panel


def get_stock_series(rv_df, ticker):
    s = rv_df[ticker]
    # Identify the stock's own start/end (first/last non-NaN)
    valid = s.dropna()
    if len(valid) == 0:
        raise ValueError(f"No data for ticker {ticker}")
    stock_start = valid.index[0]
    stock_end   = valid.index[-1]
    print(f"  [{ticker}] Data: {stock_start.date()} to {stock_end.date()} ({len(valid)} obs)")
    return s  # keep NaNs for global alignment

def create_lagged_features(series, lags, use_log_transform=True):
    df = pd.DataFrame({'RV_daily': series})
    df['log_RV'] = np.log(df['RV_daily'].clip(lower=1e-8))
    feature_source = df['log_RV'] if use_log_transform else df['RV_daily']
    for lag in lags:
        df[f'lag_{lag}'] = feature_source.shift(lag)
    df = df.dropna()
    return df

In [45]:
# BIC lag selection
def calculate_BIC(selection_data, max_lag: int = 60, verbose: bool = True):
    bic_values = {}
    for lag in range(1, max_lag + 1):
        model  = AutoReg(
            selection_data, lags=lag, old_names=False, hold_back=max_lag
        )
        result = model.fit()
        bic_values[lag] = result.aic

    best_lag = min(bic_values, key=bic_values.get)

    lookback = list(range(best_lag, 0, -1))
    if verbose:
        print(f"    BIC-selected lookback period: {best_lag} ({len(lookback)} lags)")
    return lookback, best_lag

In [46]:
# Training utilities
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0

def train_single_model(X_train, y_train, X_val, y_val, config, verbose=False):
    X_train_tensor = torch.FloatTensor(X_train).to(config.device)
    y_train_tensor = torch.FloatTensor(y_train).to(config.device)
    X_val_tensor   = torch.FloatTensor(X_val).to(config.device)
    y_val_tensor   = torch.FloatTensor(y_val).to(config.device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader  = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

    model     = LSTMForecaster(X_train.shape[1], config.hidden_layers).to(config.device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    early_stopping = EarlyStopping(patience=config.early_stopping_rounds)
    best_val   = float('inf')
    best_state = None
    best_epoch = 0

    for epoch in range(config.epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            out  = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / max(1, len(train_loader))

        model.eval()
        with torch.inference_mode():
            vout  = model(X_val_tensor)
            vloss = criterion(vout, y_val_tensor).item()

        if (best_val - vloss) > early_stopping.min_delta:
            best_val   = vloss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        early_stopping(vloss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}; best={best_epoch} val={best_val:.6f}")
            break

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.epochs}  Train: {avg_train_loss:.6f}  Val: {vloss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def set_seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def train_ensemble(
    X_train, y_train, X_val, y_val, config, base_seed=42, verbose=True
):
    models    = []
    if verbose:
        print(f"    Training ensemble of {config.n_ensembles} models...")
    for i in range(config.n_ensembles):
        set_seed_all(base_seed + i)
        model = train_single_model(X_train, y_train, X_val, y_val, config, verbose=False)
        models.append(model)
    return models

def predict_ensemble(models, X, config):
    X_tensor    = torch.FloatTensor(X).to(config.device)
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())
    return np.stack(predictions, axis=0)

In [47]:
# Metrics
def calculate_mse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred)

def calculate_qlike(y_true_vol, y_pred_vol):
    """Variance QLIKE computed from volatility-level observations and forecasts."""
    y_true_var = np.square(np.maximum(np.asarray(y_true_vol, dtype=float), 1e-8))
    y_pred_var = np.square(np.maximum(np.asarray(y_pred_vol, dtype=float), 1e-8))
    ratio = y_true_var / y_pred_var
    return np.mean(ratio - np.log(ratio) - 1)

In [48]:
# Per-stock expanding window forecast
def _expanding_window_forecast_single_sequential(
    ticker: str,
    rv_series: pd.Series,   # raw RV values
    global_index: pd.DatetimeIndex,
    config,
    only_test_start=None,
    window_seed=None,
    verbose=True
):
    # build lagged feature frame for this stock
    stock_rv = rv_series.dropna()          # restrict to the stock's own date range

    if config.use_bic:
        df_with_lags = create_lagged_features(
            stock_rv,
            list(range(1, config.max_bic_lag + 1)),
            use_log_transform=config.use_log_transform
        )
    else:
        df_with_lags = create_lagged_features(
            stock_rv,
            config.lag_candidates,
            use_log_transform=config.use_log_transform
        )
        feature_cols = [col for col in df_with_lags.columns if col.startswith('lag_')]

    # expanding window setup
    start_date        = df_with_lags.index[0]
    initial_train_end = start_date + pd.DateOffset(years=config.initial_train_years)
    current_test_start = initial_train_end + pd.DateOffset(years=config.validation_years)
    if only_test_start is not None:
        current_test_start = pd.Timestamp(only_test_start)

    # Storage: one value per global date
    pred_dict = {}   # date -> predicted RV
    bic_dict  = {}   # date -> chosen lookback period (int)

    window_count = 0

    while current_test_start <= df_with_lags.index[-1]:
        window_count += 1

        train_end = current_test_start - pd.DateOffset(years=config.validation_years)
        next_test_start = current_test_start + pd.DateOffset(months=config.test_months)

        # Half-open validation and test windows prevent boundary overlap:
        # validation = (train_end, current_test_start)
        # test       = [current_test_start, next_test_start)
        train_data = df_with_lags.loc[
            (df_with_lags.index >= start_date)
            & (df_with_lags.index <= train_end)
        ]
        val_data = df_with_lags.loc[
            (df_with_lags.index > train_end)
            & (df_with_lags.index < current_test_start)
        ]
        test_data = df_with_lags.loc[
            (df_with_lags.index >= current_test_start)
            & (df_with_lags.index < next_test_start)
        ]

        # Keep memory selection explicit. This dedicated slice covers the most
        # recent configured number of years before the nominal test start.
        bic_selection_start = (
            current_test_start - pd.DateOffset(years=config.bic_selection_years)
        )
        bic_selection_data = df_with_lags.loc[
            (df_with_lags.index >= bic_selection_start)
            & (df_with_lags.index < current_test_start)
        ].copy()

        if (len(test_data) == 0 or len(train_data) == 0 or len(val_data) == 0
                or (config.use_bic and len(bic_selection_data) == 0)):
            break

        if verbose:
            print(f"  [{ticker}] Window {window_count}: "
                  f"train {train_data.index[0].date()}–{train_data.index[-1].date()} "
                  f"| val {val_data.index[0].date()}–{val_data.index[-1].date()} "
                  f"| test {test_data.index[0].date()}–{test_data.index[-1].date()} "
                  f"({len(test_data)} obs)")

        # lag selection
        if config.use_bic:
            bic_lags, best_lag_int = calculate_BIC(
                bic_selection_data['log_RV'], config.max_bic_lag, verbose=verbose
            )
            current_feature_cols   = [f'lag_{lag}' for lag in bic_lags]
        else:
            current_feature_cols = feature_cols
            best_lag_int         = None

        # features & targets
        X_train = train_data[current_feature_cols].values
        X_val   = val_data[current_feature_cols].values
        X_test  = test_data[current_feature_cols].values

        y_train_original = train_data['RV_daily'].values.reshape(-1, 1)
        y_val_original   = val_data['RV_daily'].values.reshape(-1, 1)
        y_test_original  = test_data['RV_daily'].values.reshape(-1, 1)

        if config.use_log_transform:
            y_train = np.log(np.maximum(y_train_original, 1e-8))
            y_val   = np.log(np.maximum(y_val_original,   1e-8))
        else:
            y_train = y_train_original.copy()
            y_val   = y_val_original.copy()

        # scaling
        feature_scaler  = StandardScaler()
        X_train_scaled  = feature_scaler.fit_transform(X_train)
        X_val_scaled    = feature_scaler.transform(X_val)
        X_test_scaled   = feature_scaler.transform(X_test)

        if config.target_scaler:
            target_scaler   = StandardScaler()
            y_train_scaled  = target_scaler.fit_transform(y_train)
            y_val_scaled    = target_scaler.transform(y_val)
        else:
            y_train_scaled  = y_train
            y_val_scaled    = y_val
            target_scaler   = None

        # train & predict
        models = train_ensemble(
            X_train_scaled,
            y_train_scaled,
            X_val_scaled,
            y_val_scaled,
            config,
            base_seed=config.random_seed if window_seed is None else window_seed,
            verbose=verbose
        )

        test_pred_scaled_all = predict_ensemble(models, X_test_scaled, config)

        if config.target_scaler and target_scaler is not None:
            test_log_all = np.stack(
                [target_scaler.inverse_transform(p) for p in test_pred_scaled_all], axis=0
            )
        else:
            test_log_all = test_pred_scaled_all

        if config.use_log_transform:
            test_pred = np.mean(np.exp(test_log_all), axis=0)
        else:
            test_pred = np.mean(test_log_all, axis=0)

        test_pred = np.maximum(test_pred, 1e-12).flatten()

        # store predictions aligned to global index
        for i, date in enumerate(test_data.index):
            pred_dict[date] = test_pred[i]
            if config.use_bic:
                bic_dict[date] = best_lag_int   # same lookback chosen for whole test window

        current_test_start = next_test_start
        if only_test_start is not None:
            break

    # reindex to global calendar
    pred_series = pd.Series(pred_dict, name=ticker).reindex(global_index)
    bic_series  = pd.Series(bic_dict,  name=ticker).reindex(global_index)

    return pred_series, bic_series

In [49]:
# Portable CPU parallelism: one complete ensemble per window job.
def format_duration(seconds):
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def forecast_window_job(
    ticker,
    window_count,
    test_start,
    rv_series,
    global_index,
    config
):
    warnings.filterwarnings('ignore')
    torch.set_num_threads(config.torch_threads_per_worker)

    # Stable across worker counts and scheduling order.
    ticker_seed = zlib.crc32(ticker.encode('utf-8'))
    seed_limit = 2**31 - 1
    window_seed = (
        config.random_seed
        + ticker_seed
        + window_count * config.n_ensembles
    ) % seed_limit

    pred_s, bic_s = _expanding_window_forecast_single_sequential(
        ticker=ticker,
        rv_series=rv_series,
        global_index=global_index,
        config=config,
        only_test_start=test_start,
        window_seed=window_seed,
        verbose=False
    )
    return window_count, pred_s.dropna(), bic_s.dropna()


def expanding_window_forecast_single(
    ticker: str,
    rv_series: pd.Series,
    global_index: pd.DatetimeIndex,
    config
):
    stock_rv = rv_series.dropna()
    lags = (
        list(range(1, config.max_bic_lag + 1))
        if config.use_bic
        else config.lag_candidates
    )
    df_with_lags = create_lagged_features(
        stock_rv,
        lags,
        use_log_transform=config.use_log_transform
    )

    start_date = df_with_lags.index[0]
    test_start = (
        start_date
        + pd.DateOffset(years=config.initial_train_years)
        + pd.DateOffset(years=config.validation_years)
    )
    window_starts = []
    while test_start <= df_with_lags.index[-1]:
        window_starts.append(test_start)
        test_start += pd.DateOffset(months=config.test_months)

    print(
        f"  [{ticker}] Dispatching {len(window_starts)} windows to "
        f"{config.cpu_workers} CPU workers"
    )
    results = Parallel(
        n_jobs=config.cpu_workers,
        backend='loky',
        batch_size=1,
        return_as='generator_unordered',
        verbose=config.parallel_verbose
    )(
        delayed(forecast_window_job)(
            ticker,
            window_count,
            current_test_start,
            rv_series,
            global_index,
            config
        )
        for window_count, current_test_start
        in enumerate(window_starts, start=1)
    )

    pred_dict = {}
    bic_dict = {}
    completed_windows = 0
    total_windows = len(window_starts)
    started_at = time.perf_counter()
    for window_count, pred_s, bic_s in results:
        completed_windows += 1
        elapsed = time.perf_counter() - started_at
        eta = (elapsed / completed_windows) * (total_windows - completed_windows)

        if pred_s.empty:
            print(
                f"  [{ticker}] {completed_windows}/{total_windows} complete | "
                f"window {window_count} produced no predictions | "
                f"elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}",
                flush=True
            )
            continue
        pred_dict.update(pred_s.to_dict())
        bic_dict.update(bic_s.to_dict())

        actual_s = rv_series.reindex(pred_s.index)
        metric_mask = np.isfinite(actual_s.values) & np.isfinite(pred_s.values)
        if metric_mask.any():
            actual_values = actual_s.values[metric_mask]
            predicted_values = pred_s.values[metric_mask]
            window_mse = calculate_mse(actual_values, predicted_values)
            window_qlike = calculate_qlike(actual_values, predicted_values)
            metric_status = (
                f" | MSE {window_mse:.6g} | QLIKE {window_qlike:.6g}"
            )
        else:
            metric_status = " | MSE n/a | QLIKE n/a"

        bic_status = ""
        if config.use_bic and not bic_s.empty:
            bic_status = f" | BIC lag {int(bic_s.iloc[0])}"
        print(
            f"  [{ticker}] {completed_windows}/{total_windows} complete | "
            f"window {window_count} | "
            f"test {pred_s.index[0].date()}-{pred_s.index[-1].date()} "
            f"({len(pred_s)} obs){bic_status}{metric_status} | "
            f"elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}",
            flush=True
        )

    print(
        f"  [{ticker}] Completed {completed_windows}/{total_windows} windows "
        f"in {format_duration(time.perf_counter() - started_at)}"
    )
    pred_series = pd.Series(pred_dict, name=ticker).reindex(global_index)
    bic_series = pd.Series(bic_dict, name=ticker).reindex(global_index)
    return pred_series, bic_series

In [50]:
# Structured output helpers and all-stock runner

def get_lstm_output_layout(config):
    mode_tag = 'BIC' if config.use_bic else str(config.lookback_period)
    model_name = f'LSTM_{mode_tag}'
    output_dir = Path(config.output_root) / model_name
    individual_dir = output_dir / 'individual'
    return model_name, output_dir, individual_dir


def save_individual_stock_output(
    ticker,
    pred_s,
    bic_s,
    actual_s,
    individual_dir,
    config
):
    individual_dir.mkdir(parents=True, exist_ok=True)
    detail_df = pd.concat(
        [actual_s.rename('real'), pred_s.rename('predicted')], axis=1
    )
    valid_mask = (
        np.isfinite(detail_df['real'].to_numpy())
        & np.isfinite(detail_df['predicted'].to_numpy())
    )
    detail_df = detail_df.loc[valid_mask].copy()
    detail_df.insert(0, 'ticker', ticker)
    detail_df['mse'] = (
        detail_df['real'] - detail_df['predicted']
    ) ** 2
    actual_variance = np.square(detail_df['real'].clip(lower=1e-8))
    predicted_variance = np.square(detail_df['predicted'].clip(lower=1e-8))
    ratio = actual_variance / predicted_variance
    detail_df['qlike'] = ratio - np.log(ratio) - 1.0

    if config.use_bic:
        detail_df['lookback_period'] = pd.to_numeric(
            bic_s.reindex(detail_df.index), errors='coerce'
        ).astype('Int64')

    detail_df.index.name = 'date'
    detail_df = detail_df.reset_index()
    detail_columns = ['ticker', 'date', 'real', 'predicted', 'mse', 'qlike']
    if config.use_bic:
        detail_columns.append('lookback_period')
    detail_df = detail_df[detail_columns]
    safe_ticker = ''.join(
        char if char.isalnum() or char in '-_.' else '_'
        for char in str(ticker)
    )
    individual_path = individual_dir / f'{safe_ticker}.csv'
    detail_df.to_csv(individual_path, index=False)
    return individual_path


def run_all_stocks(rv_df: pd.DataFrame, config, individual_dir) -> tuple:
    global_index = rv_df.index
    tickers      = rv_df.columns.tolist()

    pred_cols = {}
    bic_cols  = {}

    for ticker in tickers:
        print(f"\n{'='*60}")
        print(f"Processing: {ticker}")
        print(f"{'='*60}")
        try:
            pred_s, bic_s = expanding_window_forecast_single(
                ticker      = ticker,
                rv_series   = rv_df[ticker],
                global_index= global_index,
                config      = config
            )
            pred_cols[ticker] = pred_s
            bic_cols[ticker]  = bic_s
            individual_path = save_individual_stock_output(
                ticker=ticker,
                pred_s=pred_s,
                bic_s=bic_s,
                actual_s=rv_df[ticker].reindex(global_index),
                individual_dir=individual_dir,
                config=config
            )
            print(f"  [{ticker}] Saved individual output: {individual_path}")
        except Exception as e:
            print(f"  [{ticker}] FAILED: {e}")
            pred_cols[ticker] = pd.Series(np.nan, index=global_index, name=ticker)
            bic_cols[ticker]  = pd.Series(np.nan, index=global_index, name=ticker)

    pred_df = pd.DataFrame(pred_cols, index=global_index)
    bic_df  = pd.DataFrame(bic_cols,  index=global_index)

    return pred_df, bic_df

In [51]:
# Main entry point

def build_overall_evaluation(pred_df, input_df, model_name):
    actual_df = input_df.reindex(index=pred_df.index, columns=pred_df.columns)
    actual_values = actual_df.to_numpy(dtype=float)
    predicted_values = pred_df.to_numpy(dtype=float)
    valid_mask = np.isfinite(actual_values) & np.isfinite(predicted_values)

    if valid_mask.any():
        matched_actual = actual_values[valid_mask]
        matched_predicted = predicted_values[valid_mask]
        overall_mse = calculate_mse(matched_actual, matched_predicted)
        overall_qlike = calculate_qlike(matched_actual, matched_predicted)
    else:
        overall_mse = np.nan
        overall_qlike = np.nan

    evaluated_tickers = sum(
        np.any(valid_mask[:, column_index])
        for column_index in range(valid_mask.shape[1])
    )
    return pd.DataFrame([{
        'model': model_name,
        'requested_tickers': int(pred_df.shape[1]),
        'evaluated_tickers': int(evaluated_tickers),
        'observations': int(valid_mask.sum()),
        'mse': overall_mse,
        'qlike': overall_qlike,
    }])


def save_lstm_outputs(pred_df, bic_df, input_df, config):
    model_name, output_dir, individual_dir = get_lstm_output_layout(config)
    output_dir.mkdir(parents=True, exist_ok=True)

    summary_path = output_dir / f'{model_name}.csv'
    pred_df.to_csv(summary_path, index_label='date')

    evaluation_df = build_overall_evaluation(pred_df, input_df, model_name)
    evaluation_path = output_dir / f'{model_name}_evaluation.csv'
    evaluation_df.to_csv(evaluation_path, index=False)

    config_path = output_dir / f'{model_name}_config.json'
    output_paths = {
        'prediction_summary': str(summary_path),
        'evaluation': str(evaluation_path),
        'individual_directory': str(individual_dir),
        'run_config': str(config_path),
    }

    non_nan_predictions = int(pred_df.notna().sum().sum())
    run_config = {
        'model': model_name,
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'use_bic': bool(config.use_bic),
        'fixed_lookback_period': None if config.use_bic else int(config.lookback_period),
        'max_bic_lag': int(config.max_bic_lag),
        'bic_selection_years': int(config.bic_selection_years),
        'initial_train_years': int(config.initial_train_years),
        'validation_years': int(config.validation_years),
        'test_months': int(config.test_months),
        'hidden_layers': list(config.hidden_layers),
        'learning_rate': float(config.learning_rate),
        'batch_size': int(config.batch_size),
        'epochs': int(config.epochs),
        'early_stopping_rounds': int(config.early_stopping_rounds),
        'n_ensembles': int(config.n_ensembles),
        'random_seed': int(config.random_seed),
        'batch_normalization': bool(config.batch_normalization),
        'parallel_strategy': 'cpu_windows',
        'cpu_workers': int(config.cpu_workers),
        'torch_threads_per_worker': int(config.torch_threads_per_worker),
        'use_log_transform': bool(config.use_log_transform),
        'target_scaler': bool(config.target_scaler),
        'estimate_type': config.estimate_type,
        'value_column': config.value_column,
        'data_source': str(config.data_source),
        'data_files': list(config.data_files),
        'load_all_csv_files': not bool(config.data_files),
        'input_start_date': input_df.index.min().isoformat(),
        'input_end_date': input_df.index.max().isoformat(),
        'input_dates': int(input_df.shape[0]),
        'symbols': input_df.columns.tolist(),
        'symbol_count': int(input_df.shape[1]),
        'prediction_shape': list(pred_df.shape),
        'non_nan_predictions': non_nan_predictions,
        'device': str(config.device),
        'output_files': output_paths,
    }

    with config_path.open('w', encoding='utf-8') as config_file:
        json.dump(run_config, config_file, indent=2)

    return output_dir, output_paths

def main_lstm():
    print("="*60)
    print("LSTM NEURAL NETWORK – ALL STOCKS")
    print("="*60)

    config = Config_LSTM()
    print(f"\nUsing device: {config.device}")
    print(f"BIC enabled:  {config.use_bic}")
    model_name, output_dir, individual_dir = get_lstm_output_layout(config)
    individual_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output model: {model_name}")

    # Load data
    print(f"\nLoading data from: {config.data_source}")
    rv_df = load_dacheng_xiu_panel(
        config.data_source,
        config.data_files,
        estimate_type=config.estimate_type,
        value_column=config.value_column
    )

    # Run forecasts
    pred_df, bic_df = run_all_stocks(rv_df, config, individual_dir)

    # Save output
    output_dir, output_paths = save_lstm_outputs(pred_df, bic_df, rv_df, config)
    print(f"\nSaved model outputs to: {output_dir}")
    for output_name, output_path in output_paths.items():
        print(f"  {output_name}: {output_path}")

    print(f"\nOutput shape: {pred_df.shape[0]} dates x {pred_df.shape[1]} stocks")
    non_nan = pred_df.notna().sum().sum()
    print(f"Total predictions made: {non_nan:,}")

    return pred_df, bic_df

if __name__ == '__main__':
    pred_df, bic_df = main_lstm()

LSTM NEURAL NETWORK – ALL STOCKS

Using device: cpu
BIC enabled:  False
Output model: LSTM_10

Loading data from: ..\Data
Constructed panel from 1 Dacheng Xiu CSV file(s) using QMLE-Trade; Volatility values were not squared
Panel shape: 7646 dates x 1 symbols
Global date range: 1996-01-02 to 2026-06-30

Processing: AAPL
  [AAPL] Dispatching 294 windows to 8 CPU workers
  [AAPL] 1/294 complete | window 2 | test 2002-02-19-2002-03-15 (19 obs) | MSE 0.0152153 | QLIKE 0.1269 | elapsed 00:00:07 | ETA 00:38:03
  [AAPL] 2/294 complete | window 6 | test 2002-06-17-2002-07-15 (20 obs) | MSE 0.033838 | QLIKE 0.264112 | elapsed 00:00:08 | ETA 00:19:36
  [AAPL] 3/294 complete | window 8 | test 2002-08-16-2002-09-13 (20 obs) | MSE 0.00734967 | QLIKE 0.0860394 | elapsed 00:00:08 | ETA 00:13:11
  [AAPL] 4/294 complete | window 1 | test 2002-01-16-2002-02-15 (22 obs) | MSE 0.0120521 | QLIKE 0.146468 | elapsed 00:00:09 | ETA 00:11:21
  [AAPL] 5/294 complete | window 3 | test 2002-03-18-2002-04-15 (20 o